# Read labels
- This is some code to read the output of the label tool and plot the data with the associated maxima and minima.

In [ ]:
import json
import sys
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from helpers.df_ops import prepare_df

LABELS_PATH = Path("labels.json")
DATA_DIR    = PROJECT_ROOT / "Data"
TOL         = 6

def find_file(fname):
    matches = list(DATA_DIR.rglob(fname))
    if not matches:
        raise FileNotFoundError(f"{fname} not found under {DATA_DIR}")
    return matches[0]

def load_raw(fname):
    raw_df = pd.read_csv(find_file(fname), sep=r'\s+', skip_blank_lines=True)
    try:
        raw_data_df = prepare_df(raw_df, add_prefix=False, relative=True)
    except Exception:
        raw_data_df = prepare_df(raw_df, add_prefix=True, relative=True)
    day_offset = raw_data_df['day'].min()
    raw_data_df = raw_data_df.copy()
    raw_data_df['day'] -= day_offset
    return raw_data_df, day_offset

with open(LABELS_PATH) as f:
    labels = json.load(f)

print(f"Loaded labels for {len(labels)} files:")
for fname, lbl in labels.items():
    is_const = lbl.get('const', False)
    tag = '  [CONST]' if is_const else ''
    print(f"  {fname}: {len(lbl['maxima'])} maxima, {len(lbl['minima'])} minima{tag}")

In [ ]:
# # Migrate labels.json to offset-corrected coordinates (run once).
# # Skips files already marked as migrated.

# migrated_any = False

# for fname, lbl in labels.items():
#     if lbl.get('_offset_applied'):
#         continue
#     _, day_offset = load_raw(fname)
#     if day_offset != 0:
#         lbl['maxima'] = [v - day_offset for v in lbl['maxima']]
#         lbl['minima'] = [v - day_offset for v in lbl['minima']]
#         print(f"  {fname}: subtracted offset {day_offset:.4f}")
#     lbl['_offset_applied'] = True
#     migrated_any = True

# if migrated_any:
#     with open(LABELS_PATH, 'w') as f:
#         json.dump(labels, f, indent=2)
#     print("labels.json updated.")
# else:
#     print("All files already migrated — nothing to do.")

  HD160346_Mt_wilson_data.txt: subtracted offset 2439670.8000
  HD201091_Mt_wilson_data.txt: subtracted offset 2439670.8000
  HD81809_Mt_wilson_data.txt: subtracted offset 2439194.8000
labels.json updated.


In [ ]:
def load_clean(fname):
    raw_data_df, _ = load_raw(fname)
    med = raw_data_df['sind'].median()
    mad = (raw_data_df['sind'] - med).abs().median()
    return (
        raw_data_df[(raw_data_df['sind'] - med).abs() < TOL * mad]
        .sort_values('day')
        .reset_index(drop=True)
    )

fig, axes = plt.subplots(len(labels), 1, figsize=(20, 5 * len(labels)))
if len(labels) == 1:
    axes = [axes]

for ax, (fname, lbl) in zip(axes, labels.items()):
    data_df  = load_clean(fname)
    is_const = lbl.get('const', False)

    ax.plot(data_df['day'], data_df['sind'], '.', color='steelblue',
            markersize=3, alpha=0.7, rasterized=True)

    if is_const:
        ax.set_facecolor('#fff8e1')
        ax.text(0.5, 0.5, 'CONST', transform=ax.transAxes,
                fontsize=28, color='#ffb300', alpha=0.4,
                ha='center', va='center', fontweight='bold')
    else:
        y_max = data_df['sind'].max()
        y_min = data_df['sind'].min()
        y_pad = (y_max - y_min) * 0.03

        for day in lbl['maxima']:
            ax.axvline(day, color='royalblue', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.text(day, y_max + y_pad, 'MAX', color='royalblue',
                    fontsize=7, ha='center', va='bottom', rotation=90)

        for day in lbl['minima']:
            ax.axvline(day, color='darkorange', linestyle='--', linewidth=1.2, alpha=0.8)
            ax.text(day, y_min - y_pad, 'MIN', color='darkorange',
                    fontsize=7, ha='center', va='top', rotation=90)

    title_tag = '  [CONST]' if is_const else ''
    ax.set_title(f"{fname}{title_tag}")
    ax.set_xlabel("Days since first observation")
    ax.set_ylabel("S-index")

plt.tight_layout()
plt.show()